<div>
<center><img src="../assets/Flux-logo.svg" width="360"/></center>
</div>

<div style="background:linear-gradient(90deg,#036291 0%,#91C2D8 100%);padding:20px 26px;border-radius:10px;border-left:10px solid #D9A441;margin-top:18px">
<h1 style="margin:0;color:#ffffff">Module 1: Coscheduling Quantum and Classical Work with QRMI</h1>
<p style="margin:6px 0 0 0;color:#DCECF4;font-size:15px">One flux submit generates a classical reservation and a scout that will open the vendor session that will release the held reservation.</p>
<p style="margin:2px 0 0 0;color:#DCECF4;font-size:13px">flux-quantum &middot; QRMI &middot; IBM Quantum</p>
</div>


# Welcome

This hpc7g.16xlarge instance runs a single node Flux instance with the [flux-quantum](https://github.com/converged-computing/flux-quantum) plugin, and a pipeline you can test first without a token, on a mock vendor, and then run as one cheap coscheduled job against an IBM QPU through [QRMI](https://github.com/qiskit-community/qrmi), the Quantum Resource Management Interface. Feel free to tweak it and run a more expensive one, if you please.

| component     | version                                  | why?                                    |
|---------------|------------------------------------------|-----------------------------------------|
| ubuntu        | 24.04, python 3.12                       | qrmi needs python 3.11 or newer         |
| flux-security | 0.15.0                                   | release tarball                         |
| flux-core     | 0.87.0                                   | release tarball, prefix /usr            |
| flux-sched    | vsoch/flux-sched branch `hold`           | git, cmake, sysconfdir /etc             |
| flux-quantum  | branch `braket-hybrid`                   | git clone like the cluster, editable    |
| qrmi          | latest `qrmi[ibm]` on PyPI, with qiskit  | pip                                     |
| ibmcloud cli  | latest                                   | clis.cloud.ibm.com                      |

This is not a system instance under systemd. Keep in mind that would be part of a production setup, and is part of the setup where this is being developed. The instance is started with `flux start`, run as the `ubuntu` user, and the scheduler is configured through `--config-path` with the same two config tables the cluster writes to `/etc/flux/system/conf.d`. Everything the scout and the classical job do is the same.

## Set up your workspace

Click on <button data-commandLinker-command="terminal:open" data-name="flux" href="#">this button</button> to create a Terminal. Drag it to sit alongside this notebook, so you have the terminal on the left and this text on the right.

Every command in this notebook is meant to be **copied into that terminal**. None of the cells here run. The terminal JupyterLab opens is already inside the Flux instance, with `FLUX_URI` set, because JupyterLab is the instance's initial program.

```bash
cd ~/tutorial/module1
```

<div class="alert alert-block" style="background-color:#DCECF4;color:#06293D">
<span style="font-weight:600">Tip:</span> The scripts the notebook uses, <code>flux-quantum-check</code>, <code>flux-quantum-demo</code>, and <code>ibm-run</code>, are installed in <code>/usr/local/bin</code>. Their sources are under <code>/opt/flux-quantum/examples/qrmi</code> if you want to read them.
</div>


# The design

Let's talk about the coscheduling design. We add a flux-quantum client plugin, which is an interface to customize flux commands for submit, run, batch, and so on. With flux-quantum we can add flags however we like that say "I want to run a quantum job." Our strategy is simple. We take a request for hybrid quantum work (classical and quantum resources) and split it into a hybrid pair.

- A **classical job** that is "held" until specifically told to release (a form of reservation)
- A **scout job**, or probe, that creates a quantum session, monitors it, and releases the
classical resources when it is ready

With this approach, no classical resources are running until the quantum is ready. Smaller jobs can backfill into the space with lower priority that can be pre-empted. The design uses priority and ensures that we do not accept more hybrid jobs than could be supported given the size of the cluster resources and contender pre-emptible jobs. Full experiments to demonstrate the approach are pending, we have only mocked the setup due to funding limitations.

The two halves never share a filesystem. The handoff is a `memo` event on the classical job's eventlog with the session id, which the job's environment picks up as `QUANTUM_SESSION_ID` and under the names QRMI's own job-side contract uses. That is why the two halves can land on different nodes.


## 1. Look around the instance

You are in a Flux instance. Take a look.

```bash
flux resource list
```

That is the 64 cores of the hpc7g.16xlarge. Take a look to see that fluxion is handling scheduling.

```bash
flux module list
```

And that we have the quantum plugin loaded.

```bash
flux jobtap list
```
```bash
flux jobtap query quantum.so | jq
```

<div class="alert alert-block" style="background-color:#91C2D8;color:#06293D">
<span style="font-weight:600">Description:</span> The jobtap plugin is for validation. It ensures that we can support a vendor, that the request is feasible given the size of the cluster, and it restores its state when Flux reloads it.
</div>

JupyterLab was started on this instance with `flux-quantum-start`, which writes a config and runs `flux start --config-path=... flux-quantum-init jupyter lab`. The config is what loads the jobtap plugin and configures the job manager, and `flux-quantum-init` populates the vendor devices into the fluxion graph before anything else runs. Read the config it wrote.

```bash
cat ~/.flux-quantum/conf.d/coschedule.toml
```

We use a `coschedule` queue policy that lets a held job reserve. Under `fcfs` a held job is skipped and never reserves, which is the one policy where coscheduling silently does not work.


## 2. Check our integrations

`flux-quantum-check` runs all of the checks below in one go. Do them by hand first, so it is clear what each one is asking. Every piece has to be there before a pair can work.

**The python flux uses, and the plugin in it.** `flux python` is the interpreter flux was built against, and it is where flux_quantum, qrmi, and qiskit are installed. On this instance plain `python3` is the same interpreter, but `flux python` is the form that works everywhere.

```bash
flux version
flux python -c "import sys; print(sys.version.split()[0], sys.executable)"
flux python -c "import flux_quantum; print(flux_quantum.__file__)"
```

**The scheduler is fluxion, with the coschedule policy.** sched-simple cannot hold a job, and under fluxion's default `fcfs` a held job is skipped and never reserves.

```bash
flux module list | grep sched-fluxion
flux config get sched-fluxion-qmanager.queue-policy
```

**The jobtap plugin is loaded.** You saw its query in the last section. This is the short form.

```bash
flux jobtap list
```

**The cli plugin is discovered.** flux finds it through `FLUX_CLI_PLUGINPATH`, which the image sets. If the `--quantum` options are missing from the help, nothing below will parse.

```bash
echo $FLUX_CLI_PLUGINPATH
flux submit --help | grep -- --quantum
```
Seven options should come back. `--quantum-vendor`, `--quantum-select`, `--quantum-device`, `--quantum-hold`, `--quantum-hold-max`, `--quantum-wait`, and `--quantum-dry-run`. This notebook uses the vendor, device, and wait ones.

**The vendor devices are in the graph.** `flux-quantum-init` populated one `qdevice` vertex per vendor before anything ran. The scout is allocated one of these for as long as the pair lives, which is what makes a QPU a scheduled resource. This is the same find RPC populate uses.

```bash
flux python -c "
import flux
from flux_quantum import graph
g = graph.get_live_graph(flux.Flux())
print(sorted(graph.vendors_present(g)))"
```

**qrmi imports, with qiskit.** The IBM backend still speaks the pre-0.23 QRMI names, which qrmi keeps as deprecated aliases, so the second line should say `True`.

```bash
flux python -c "import qrmi, qiskit; print('qrmi', getattr(qrmi, '__version__', '?'), 'qiskit', qiskit.__version__)"
flux python -c "from qrmi import ResourceType; print(hasattr(ResourceType, 'IBMQiskitRuntimeService'))"
ibmcloud version
```

**The mock vendor is registered.** It only is when this variable was set before `flux start`, because the ingest validator is a separate process the broker started.

```bash
echo $FLUX_QUANTUM_MOCK
```

Now the whole thing at once. Each line says what it found, and a hard failure makes the exit code nonzero.

```bash
flux-quantum-check
```

<div class="alert alert-block" style="background-color:#DCECF4;color:#06293D">
<span style="font-weight:600">Tip:</span> <code>flux-quantum-check --ibm</code> adds one more check, the IBM credentials by name. You will run that one by hand in section 4.
</div>


## 3. Submit a Mock Hybrid Pair

<div class="alert alert-block" style="background-color:#91C2D8;color:#06293D">
<span style="font-weight:600">Description:</span> The mock vendor is a way to test without incurring charges or using credits. It opens a session, simulates a queue, and hands back a session id.
</div>

### Submit

First, slow the mock down so there is time to watch. The mock simulates a vendor queue, a fixed 10 seconds of overhead plus one service time per task ahead of us. With 20 tasks ahead and one second each, the scout will wait about 30 seconds before it releases the classical job.

```bash
export FLUX_QUANTUM_MOCK_QUEUE=20          # tasks ahead of us in the vendor queue
export FLUX_QUANTUM_MOCK_SERVICE_TIME=1    # seconds per task, 0.1 by default
```

Now submit. `-t` is the classical job's time limit, and its clock starts when the scout releases the job. `--shots` only matters on a real device, the mock workload just reports its session. The workload also lingers for 20 seconds after its work (`--linger`, 20 by default) so the classical job can be seen running.

```bash
flux submit -t 5m --quantum-vendor mock -n1 -- flux python ~/tutorial/module1/workload.py --shots 100
```

`flux jobs -a` lists pending jobs before running ones, so the classical reservation is on top in `SCHED` and the scout is at the bottom in `RUN`, even though the scout was submitted second. `flux job last` is the scout, the most recent submit.

```bash
flux jobs -a
flux resource list
```

For the next 30 seconds or so the scout is `RUN` and the classical is `SCHED`. It is not waiting for cores, it has them reserved. It is waiting to be released. One core is allocated, the scout's, and the classical's core shows as free, because reserved is not allocated. When the scout releases it the classical goes to `RUN` for about 20 seconds, and when both finish they are completed, `CD`.

### Looking at the scout

Attach to the scout. It is the bottom job in the list and the most recent submit, so `flux job last` finds it. The scout is only submitted once the classical reservation is accepted, so there is never a scout without a job to release.

```bash
flux job attach $(flux job last)
```

Read it top to bottom. The mock reports its queue position as it drains. Then the scout says it opened the session, posted it to the classical job, and released the hold. The scout is **kept running** until the classical finishes, and only then closes the session. This detail ensures that we do not go over the job limit, but we close early if the job finishes before the limit, and we also can guarantee the session closes if the job crashes. In the scheduler, we account for the classical resources and the scout core, and do not submit more pairs than would be allowed based on free cores and preemptible cores.

### Looking at the classical job

As soon as the scout says `unheld`, the classical job is running. Attach to it and watch.

```bash
flux job attach <classical-job-id>
```

The job receives the session ID to use via a job memo, and it is passed in the environment before running the workload (`QUANTUM_SESSION_ID`). The first thing [`workload.py`](workload.py) does is print everything quantum related in its environment, by name. Under the mock there is no resource, so `QRMI_JOB_QPU_RESOURCES` is unset and there is nothing to submit. On a real vendor those two name the QPU, and a `<resource>_QRMI_JOB_ACQUISITION_TOKEN` line has the session id again under the name QRMI reads. The workload prints that one as `handle`, because that is what it is. QRMI's name for the handle on the open session, not a credential.

### The memo to exchange information

You can see the memo in the eventlog for the classical job. It is added by the scout right before the scout releases the classical nodes.

```bash
flux job eventlog <classical-job-id>
```

Four different parties wrote into that log.

- **`jobspec-update` with `protected`** is the flux-quantum jobtap plugin, `quantum.so`. Both halves of a pair are marked protected so they are never the ones preempted.
- **`memo`** is the scout, through the job-manager's memo RPC, which is authorized against the job owner. `quantum_session` is the handoff. `released` is read by fluxion's queue manager after a restart, so a job that was already released is not held a second time.
- **`alloc` annotations under `user`** are flux-core's job manager echoing the memo. A memo updates the job's user annotations, and the alloc event records the annotations as they stood when the job was allocated. Nothing in flux-quantum writes those.
- **`alloc` annotations under `sched`**, `selection_type: reserved`, is fluxion. It says this job was placed from the reservation it held, not scheduled fresh.

The `memo` event has `quantum_session` and it comes **before** `alloc`. That memo is the whole (what we might call) handoff. We do not use a shared filesystem, or a socket. The user is responsible for owning credentials, either in the job environment (signed as it passes through the scheduler) or other means. You can look at the resources for each.

```bash
flux job info <scout-id> R  | jq
flux job info <classical-id> R | jq
```

### Run the demo automatically

`flux-quantum-demo mock` runs the sequence above with the ids captured for you. Purge the inactive jobs first, so the listing only shows this run.

```bash
flux job purge --force --age-limit=0
flux-quantum-demo mock
```

For a quick run with no queue at all.

```bash
FLUX_QUANTUM_MOCK_QUEUE=0 flux submit -t 5m --quantum-vendor mock -n1 -- flux python ~/tutorial/module1/workload.py --linger 0
```

<div class="alert alert-block" style="background-color:#DCECF4;color:#06293D">
<span style="font-weight:600">Notice:</span> Nothing classical is running while the scout waits. That is the point of the design. The cores will not run until the vendor says the QPU is ready.
</div>

We are next going to run a real session, so put the mock's knobs away.
```bash
unset FLUX_QUANTUM_MOCK_QUEUE FLUX_QUANTUM_MOCK_SERVICE_TIME FLUX_QUANTUM_MOCK_SESSION
```


## 4. IBM credentials, by name

QRMI reads its configuration from the environment. Export the variables in this terminal. Flux copies your environment into both halves of the pair, and the ingest validator checks, by name, that the variables the chosen resource needs are in the job's environment. It never reads the values and has no credentials of its own.

You usually need to log in to ibmcloud. The passcode login can stop accepting credentials, in which case copy an API key from the IBM Cloud interface and use that.

```bash
export IBM_CLOUD_TOKEN=...
ibmcloud login --apikey "$IBM_CLOUD_TOKEN"
export IBM_CLOUD_CRN=$(ibmcloud resource service-instances --service-name quantum-computing --output json | jq -r '.[0].crn')
```

<div class="alert alert-block" style="background-color:#DCECF4;color:#06293D">
<span style="font-weight:600">Notice:</span> The credentials live only in this terminal's environment. A second terminal opened from the sidebar will not have them unless you export them there too.
</div>

### Choose a backend

List the operational devices and how deep their queues are.

```bash
flux python -c "
from qiskit_ibm_runtime import QiskitRuntimeService
import os
svc = QiskitRuntimeService(channel='ibm_quantum_platform',
                           token=os.environ['IBM_CLOUD_TOKEN'], instance=os.environ['IBM_CLOUD_CRN'])
for b in svc.backends(operational=True, simulator=False):
    print(b.name, b.status().pending_jobs)"
```

### The four QRMI variables

The convention seems to be that every QRMI variable is prefixed with the resource id, so for `ibm_kingston` the api key lives in `ibm_kingston_QRMI_IBM_QRS_IAM_APIKEY`. Four are required. This is what `flux-quantum-demo ibm` derives from `RESOURCE`, `IBM_CLOUD_TOKEN`, and `IBM_CLOUD_CRN`. Here you set them yourself.

```bash
export RESOURCE=ibm_kingston
export ${RESOURCE}_QRMI_IBM_QRS_ENDPOINT=https://quantum.cloud.ibm.com/api/v1
export ${RESOURCE}_QRMI_IBM_QRS_IAM_ENDPOINT=https://iam.cloud.ibm.com
export ${RESOURCE}_QRMI_IBM_QRS_IAM_APIKEY=$IBM_CLOUD_TOKEN
export ${RESOURCE}_QRMI_IBM_QRS_SERVICE_CRN=$IBM_CLOUD_CRN
```

**Who sets these, and who chooses the device.** flux-quantum does not set these variables. They are yours, in your environment, the same way they would be for any other QRMI client. If none are set for the device you name, the submit is refused in your shell, before anything is created. The backend will not construct, and the ingest validator, which checks the job's environment by name, refuses the jobspec. The scout never runs. A flux shell plugin, akin to the one we developed earlier, could derive them for a job, but that is not done here.

<div class="alert alert-block" style="background-color:#DCECF4;color:#06293D">
<span style="font-weight:600">Opinion (<a href="https://github.com/vsoch">@vsoch</a>):</span> the onus of setting these vendor specific variables should sit with a workflow manager, not with the user at a terminal and not with the resource manager. A workflow manager already knows which vendor, which device, and which plan a step is meant to run against, and it is the layer that composes a job's environment. Flux's job is to schedule the pair and hand off the session. Exporting four variables per device by hand is fine for a tutorial, and that is all it is.
</div>

Choosing the device is yours too, with `--quantum-device`. This mirrors the other QRMI library designs, where the user names the resource, and it assumes you have preferences about it. A plan, a queue you have watched, a device you have calibrated against. In our fluence `kubectl` plugin we added policy based selection, and `--quantum-select` here is a small version of that over the vendors you have credentials for, but in this setup the device is not selected for you.

<div class="alert alert-block" style="background-color:#DCECF4;color:#06293D">
<span style="font-weight:600">What fluxion models:</span> fluxion does model the machines, but only to enforce a quota, not to choose. <code>flux-quantum-init</code> adds one <code>qdevice_&lt;vendor&gt;</code> subtree per vendor to the graph at startup, each with a fixed number of <code>qpu</code> vertices (two by default, <code>QPUS</code>). The scout's jobspec asks for one core plus one <code>qpu</code> under its vendor, exclusively, and fluxion co-allocates both for the lifetime of the pair. So at most that many pairs can hold a vendor at once, and a scout past the quota waits in the queue like any job short of a resource. The jobtap plugin budgets the cores the same way: a pair that would exceed the machine's free cores plus one for the scout is rejected at submit. A future version could model the devices and their attributes, queue depth, calibration, cost, in the graph and let the scheduler select on them. Today the vertices have no attributes and the selection is the user's.
</div>

### Check them, by name

When we construct the backend, the environment variables are checked for, and it tells you what is missing. `flux-quantum-check --ibm` runs the same check.

```bash
flux python -c "
from flux_quantum.backends import get_backend
print(get_backend('ibm').credential_note)"
```

And this is how the plugin works out which resource you mean when `--quantum-device` is left off, the resources whose four variables are all present.

```bash
flux python -c "
from flux_quantum.backends.qrmi import resources_in_environment, missing_env
print([r for r in resources_in_environment('qiskit-runtime-service') if not missing_env(r, 'qiskit-runtime-service')])"
```

<div class="alert alert-block" style="background-color:#91C2D8;color:#06293D">
<span style="font-weight:600">Important:</span> Your plan has to allow sessions. The <code>qiskit-runtime-service</code> type opens a real session, which IBM permits on plans that support them (Premium). On an Open or Pay As You Go plan IBM answers 403, the scout will show the error and cancels the held classical job. Export <code>FLUX_QUANTUM_IBM_TYPE=ibm-quantum-system</code> for direct access instead.
</div>


## 5. The same pair against the QPU

<div class="alert alert-block" style="background-color:#DCECF4;color:#06293D">
<span style="font-weight:600">Notice:</span> This is going to incur charges or credit usage. The scout runs a one shot warm-up task to confirm the session is active, then the classical job submits a Bell circuit with 100 shots into the session. IBM bills QPU seconds used and, in dedicated mode, session time. Nothing is billed while queued.
</div>

We run the quick task because some vendors require it.

### Submit

The only differences from the mock submit are the vendor, the device, `--quantum-wait`, and `--linger 0`. The session stays open for as long as the classical job runs, so on a real device there is no reason to pay for 20 seconds of standing around.

`-t 5m` is the classical job's limit, and its clock starts when the scout releases it, not at submit. Time in IBM's queue does not count against it. The session is open from the moment the scout acquires it until the classical job finishes, so with `--linger 0` it closes as soon as the counts are back. The 5m is a ceiling on the classical run, not a hold on the session. The scout gets its own limit, the classical's plus `--quantum-wait` plus five minutes of slack, so flux does not kill it in the middle of the wait.

```bash
RESOURCE=ibm_kingston
flux submit -t 5m --quantum-vendor ibm --quantum-device $RESOURCE --quantum-wait 1800 -n1 -- flux python ~/tutorial/module1/workload.py --shots 100 --linger 0
```

Again look at the scout with `flux job attach`, and at both jobs with `flux jobs -a` and `flux job info` for `R` and the `eventlog`. While the warm-up task is in IBM's queue the scout prints the task id, then a line every thirty seconds with how long it has waited, so you can see it is alive.

<div class="alert alert-block" style="background-color:#91C2D8;color:#06293D">
<span style="font-weight:600">Description:</span> <code>--quantum-wait</code> is how long the scout waits for the warm-up task to leave IBM's queue. An IBM session goes active when its first task reaches the head of the queue, and later tasks in the session inherit that priority, so opening a session tells you little by itself. After acquiring, the scout submits the warm-up (one qubit, one measurement) and waits for it to run before releasing the classical job. If it times out, the scout closes the session and cancels the classical instead of starting it against a QPU it does not have. Waiting is free, the warm-up costs one shot.
</div>

If you catch it in time to see the job in the queue.

```bash
flux python -c "
import os
from qiskit_ibm_runtime import QiskitRuntimeService
svc = QiskitRuntimeService(channel='ibm_quantum_platform', token=os.environ['IBM_CLOUD_TOKEN'], instance=os.environ['IBM_CLOUD_CRN'])
print('pending on ibm_kingston:', svc.backend('ibm_kingston').status().pending_jobs)
for j in svc.jobs(limit=3): print(j.job_id(), j.status())"
```

### The same thing, all at once

The `flux-quantum-demo ibm` command will do the same. It derives the four QRMI variables, runs the credential check, finds the resource, asks you to confirm, and then runs the sequence above with the ids captured. `SHOTS`, `WALLTIME`, `WAIT`, and `NTASKS` tune it.

```bash
flux-quantum-demo ibm
```
```bash
SHOTS=50 WALLTIME=3m flux-quantum-demo ibm -y
```

<div class="alert alert-block" style="background-color:#DCECF4;color:#06293D">
<span style="font-weight:600">Notice:</span> The scout holds the session for as long as the classical runs. The fluxion allocation and the vendor session start and end together, so you are billed for the classical job's wall time. Set <code>-t</code> accordingly. A killed scout still releases, since the release runs in a finally block and SIGTERM is handled.
</div>


## 6. The classical job environment

Only the session id reaches the classical job, over the job eventlog. The wrapper puts it in the environment under two names, and the submit put two more there, naming the QPU.

```console
QUANTUM_SESSION_ID                       the open session, flux-quantum's own name
<resource>_QRMI_JOB_ACQUISITION_TOKEN    the same value, under the name QRMI reads
QRMI_JOB_QPU_RESOURCES                   the resource id, set at submit
QRMI_JOB_QPU_TYPES                       the resource type, set at submit
```

<div class="alert alert-block" style="background-color:#DCECF4;color:#06293D">
<span style="font-weight:600">Not a secret:</span> What the workload prints as <code>handle</code> is <code>&lt;resource&gt;_QRMI_JOB_ACQUISITION_TOKEN</code>. QRMI calls the session handle an <em>acquisition token</em>, and that is the name it reads it under, so that is the name we have to set. It is not a credential. It is the id of a session the scout opened, which does nothing without the api key, and which stops meaning anything when the scout closes the session. That is why the workload prints it and this notebook shows it in logs.
</div>

**Why the QRMI names.** QRMI defines a contract between a workload manager and the job it starts: the manager acquires the QPU, and tells the job which resource it has, of what type, and the handle on the session, through these three variables. The job side calls `get_job_qpu_resources_and_types()`, constructs a `QuantumResource`, and QRMI picks the handle up as a pre-set session instead of acquiring one of its own. QRMI's own plugins for Slurm and LSF are two implementations of the manager side of that contract. flux-quantum is a third. You would never run Flux together with either of those, the point is portability: [`workload.py`](workload.py) is written to the job side of the contract only, so the same file runs under Slurm, under LSF, or here, without knowing which workload manager started it. No acquire, no release, the scout owns the session.

Other QRMI settings, prefixed with the resource id, are read by QRMI itself.

```console
_QRMI_IBM_QRS_SESSION_MODE      batch or dedicated
_QRMI_IBM_QRS_SESSION_MAX_TTL   session lifetime
_QRMI_IBM_QRS_TIMEOUT_SECONDS   request timeout
_QRMI_IBM_QRS_SESSION_ID        join a session that already exists
```

And the knobs on the scout.

```console
--quantum-wait SECONDS                 give up after this long. Default 0, which
                                       means wait as long as the scout job may live
FLUX_QUANTUM_IBM_SKIP_WARMUP=1         release as soon as the session opens,
                                       no priority guarantee
FLUX_QUANTUM_IBM_READY_TIMEOUT=N       default 120, how long to wait for the
                                       backend to report itself up
```


<div style="background:#DCECF4;border-left:6px solid #D9A441;padding:12px 18px;color:#06293D"><strong>Module 1 complete</strong></div>

## What you covered

1. The hybrid pair. A held classical job and a scout that opens the session and releases it
2. Checking fluxion's `coschedule` policy, the jobtap and cli plugins, the devices in the graph, and qrmi
3. Running a mocked pair and viewing the memo that hands off the session
4. The QRMI environment variables
5. The same against an IBM QPU 
6. What the classical job sees, and why the names are QRMI's. The same job-side contract its Slurm and LSF plugins use, so the workload is portable across workload managers

## Resources

- [flux-quantum](https://github.com/converged-computing/flux-quantum), the plugin, with the Braket and IonQ backends alongside IBM
- [QRMI](https://github.com/qiskit-community/qrmi), the interface the IBM backend speaks
- [Flux Framework](https://flux-framework.org/) and the [Flux documentation](https://flux-framework.readthedocs.io/en/latest/)
